# Initialize TabDDPM Repo & Setup Dependencies

In [1]:
# Clone the official repository
import os
%cd /content
if not os.path.exists('/content/tab-ddpm'):
    !git clone https://github.com/yandex-research/tab-ddpm.git
%cd /content/tab-ddpm


/content
Cloning into 'tab-ddpm'...
remote: Enumerating objects: 699, done.
remote: Counting objects: 100% (326/326), done.
remote: Compressing objects: 100% (142/142), done.
remote: Total 699 (delta 205), reused 184 (delta 184), pack-reused 373 (from 1)
Receiving objects: 100% (699/699), 221.52 KiB | 1.57 MiB/s, done.
Resolving deltas: 100% (333/333), done.
/content/tab-ddpm


In [2]:
# Create an isolated Python 3.9.7 environment
MINIFORGE = '/content/miniforge3'
ENV = '/content/miniforge3/envs/tddpm'

if not os.path.exists(MINIFORGE):
    !wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -O /content/miniforge.sh
    !bash /content/miniforge.sh -b -p /content/miniforge3

if not os.path.exists(ENV):
    !/content/miniforge3/bin/conda create -y -n tddpm python=3.9.7

!/content/miniforge3/bin/conda run -n tddpm python --version


PREFIX=/content/miniforge3
Unpacking bootstrapper...
Unpacking payload...
Extracting ca-certificates-2026.7.22-hbd8a1cb_0.conda
Extracting libgomp-16.1.0-he0feb66_1.conda
Extracting libzlib-1.3.2-h25fd6f3_3.conda
Extracting nlohmann_json-abi-3.12.0-h0f90c79_2.conda
Extracting pybind11-abi-11-hc364b38_1.conda
Extracting python_abi-3.14-8_cp314.conda
Extracting tzdata-2026c-h151e31d_0.conda
Extracting _openmp_mutex-4.5-20_gnu.conda
Extracting zstd-1.5.7-hb78ec9c_7.conda
Extracting ld_impl_linux-64-2.46.1-default_hbd61a6d_102.conda
Extracting libgcc-16.1.0-ha9f2e26_1.conda
Extracting bzip2-1.0.8-hda65f42_10.conda
Extracting c-ares-1.34.8-h280c20c_1.conda
Extracting keyutils-1.6.3-h7cc23a3_1.conda
Extracting libev-4.33-h280c20c_3.conda
Extracting libexpat-2.8.1-hecca717_1.conda
Extracting libffi-3.7.0-h3435931_0.conda
Extracting libiconv-1.18-h3b78370_2.conda
Extracting liblzma-5.8.3-hb03c661_1.conda
Extracting libmpdec-4.0.0-hb03c661_2.conda
Extracting libstdcxx-16.1.0-h934c35e_1.conda
Ex

In [3]:
# Install the official PyTorch and requirements inside the isolated env
!/content/miniforge3/bin/conda run -n tddpm pip install --upgrade pip==23.3.1
!/content/miniforge3/bin/conda run -n tddpm pip install torch==1.10.1+cu111 -f https://download.pytorch.org/whl/torch_stable.html
!/content/miniforge3/bin/conda run -n tddpm pip install -r /content/tab-ddpm/requirements.txt


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 21.0 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 25.2
    Uninstalling pip-25.2:
      Successfully uninstalled pip-25.2
Looking in links: https://download.pytorch.org/whl/torch_stable.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 GB 713.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.0/57.0 kB 1.3 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of statsmodels to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.5/113.5 kB 2.8 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of matplotlib to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.6 MB/s eta 0:00:00
INFO: pip is looking at mu

# Setup Libraries

In [4]:
import random
import json

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer

# Load Dataset & Preparation

In [5]:
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI", "DiabetesPedigreeFunction", "Age", "Outcome"]

df = pd.read_csv(url, header=None, names=columns)

TARGET = "Outcome"
FEATURES = df.columns.drop(TARGET)

In [6]:
Zeros = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

X = df[FEATURES].copy()
y = df[TARGET].astype(int)

for col in Zeros:
    X[col] = X[col].replace(0, np.nan)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

In [7]:
imputer = KNNImputer(n_neighbors=5)

X_train = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=FEATURES,
    index=X_train.index
)

X_test = pd.DataFrame(
    imputer.transform(X_test),
    columns=FEATURES,
    index=X_test.index
)

In [8]:
X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train.values.astype("float32"),
    y_train.values.astype("int64"),
    test_size=0.20,
    random_state=42,
    stratify=y_train
)

In [9]:
DATA_DIR = "/content/tab-ddpm/data/pima"
os.makedirs(DATA_DIR, exist_ok=True)

np.save(f"{DATA_DIR}/X_num_train.npy", X_train_final)
np.save(f"{DATA_DIR}/X_num_val.npy", X_val)
np.save(f"{DATA_DIR}/X_num_test.npy", X_test.values.astype("float32"))

np.save(f"{DATA_DIR}/y_train.npy", y_train_final)
np.save(f"{DATA_DIR}/y_val.npy", y_val)
np.save(f"{DATA_DIR}/y_test.npy", y_test.values.astype("int64"))

with open(f"{DATA_DIR}/info.json", "w") as f:
    json.dump({
        "task_type": "binclass",
        "n_classes": 2
    }, f)

# Train Process

### Config

In [10]:
EXP_DIR = "/content/tab-ddpm/exp/pima/ddpm"

os.makedirs(EXP_DIR, exist_ok=True)

config = f"""
seed = 42
parent_dir = "{EXP_DIR}"
real_data_path = "{DATA_DIR}"

model_type = "mlp"
num_numerical_features = 8
device = "cpu"

[model_params]

is_y_cond = true
d_in = 8
num_classes = 2

[model_params.rtdl_params]

d_layers = [256, 256]
dropout = 0.0

[diffusion_params]

num_timesteps = 1000
gaussian_loss_type = "mse"
scheduler = "cosine"

[train.main]

steps = 1000
lr = 0.001
weight_decay = 1e-05
batch_size = 128

[train.T]

seed = 42
normalization = "quantile"
num_nan_policy = "__none__"
cat_nan_policy = "__none__"
cat_min_frequency = "__none__"
cat_encoding = "__none__"
y_policy = "default"

[sample]

num_samples = 1000
batch_size = 1000
seed = 42

[eval.type]

eval_model = "simple"
eval_type = "synthetic"

[eval.T]

seed = 42
normalization = "__none__"
num_nan_policy = "__none__"
cat_nan_policy = "__none__"
cat_min_frequency = "__none__"
cat_encoding = "__none__"
y_policy = "default"
"""

with open(f"{EXP_DIR}/config.toml", "w") as f:
    f.write(config)

### Tran & Generate 1000 Sample

In [11]:
%cd /content/tab-ddpm

!/content/miniforge3/bin/conda run -n tddpm \
    env PYTHONPATH=/content/tab-ddpm \
    PROJECT_DIR=/content/tab-ddpm \
    python scripts/pipeline.py \
    --config /content/tab-ddpm/exp/pima/ddpm/config.toml \
    --train \
    --sample

/content/tab-ddpm
[0]
8
{'is_y_cond': True, 'd_in': 8, 'num_classes': 2, 'rtdl_params': {'d_layers': [256, 256], 'dropout': 0.0}}
mlp
Step 500/1000 MLoss: 0.0 GLoss: 0.4551 Sum: 0.4551
Step 1000/1000 MLoss: 0.0 GLoss: 0.4417 Sum: 0.4417
mlp
Sample timestep  999
Sample timestep  998
Sample timestep  997
Sample timestep  996
Sample timestep  995
Sample timestep  994
Sample timestep  993
Sample timestep  992
Sample timestep  991
Sample timestep  990
Sample timestep  989
Sample timestep  988
Sample timestep  987
Sample timestep  986
Sample timestep  985
Sample timestep  984
Sample timestep  983
Sample timestep  982
Sample timestep  981
Sample timestep  980
Sample timestep  979
Sample timestep  978
Sample timestep  977
Sample timestep  976
Sample timestep  975
Sample timestep  974
Sample timestep  973
Sample timestep  972
Sample timestep  971
Sample timestep  970
Sample timestep  969
Sample timestep  968
Sample timestep  967
Sample timestep  966
Sample timestep  965
Sample timestep  964
Sam

### Check The Output

In [12]:
synthetic_X = np.load(
    "/content/tab-ddpm/exp/pima/ddpm/X_num_train.npy"
)

synthetic_y = np.load(
    "/content/tab-ddpm/exp/pima/ddpm/y_train.npy"
)

synthetic_tabddpm = pd.DataFrame(
    synthetic_X,
    columns=FEATURES
)

synthetic_tabddpm["Outcome"] = synthetic_y

print(synthetic_tabddpm.shape)

synthetic_tabddpm.head()

(1000, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,0.0,198.000000,122.000000,60.000000,744.000000,67.099998,2.329000,21.387747,1
1,5.0,152.845786,70.927922,25.485584,125.157344,35.336241,0.796103,51.662311,0
2,3.0,107.184849,71.023189,20.369569,120.685512,27.734885,0.373422,30.935240,0
3,0.0,198.000000,122.000000,60.000000,744.000000,67.099998,2.329000,21.000000,1
4,8.0,123.204103,84.958656,31.855343,143.199167,35.047391,0.534479,37.139310,1


In [13]:
# Post-process synthetic data to fix data types
int_columns = ['Pregnancies', 'Age', 'Outcome']

for col in int_columns:
    if col in synthetic_tabddpm.columns:
        synthetic_tabddpm[col] = synthetic_tabddpm[col].round()

display(synthetic_tabddpm.head())
print(synthetic_tabddpm.dtypes)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,0.0,198.000000,122.000000,60.000000,744.000000,67.099998,2.329000,21.0,1
1,5.0,152.845786,70.927922,25.485584,125.157344,35.336241,0.796103,52.0,0
2,3.0,107.184849,71.023189,20.369569,120.685512,27.734885,0.373422,31.0,0
3,0.0,198.000000,122.000000,60.000000,744.000000,67.099998,2.329000,21.0,1
4,8.0,123.204103,84.958656,31.855343,143.199167,35.047391,0.534479,37.0,1


Pregnancies                 float64
Glucose                     float64
BloodPressure               float64
SkinThickness               float64
Insulin                     float64
BMI                         float64
DiabetesPedigreeFunction    float64
Age                         float64
Outcome                       int64
dtype: object


In [ ]:
# Save to CSV
synthetic_tabddpm.to_csv('/content/tabddpm_raw.csv', index=False)
print("✔ Synthetic data saved")

In [15]:
import shutil
from google.colab import files

model_path = '/content/tab-ddpm/exp/pima/ddpm'
zip_name = 'tabddpm_pima_model'

if os.path.exists(model_path):
    shutil.make_archive(zip_name, 'zip', model_path)
    print(f"✔ Done")
else:
    print("❌ Error")

✔ Done


### Evaluate

In [16]:
%cd /content/tab-ddpm

!/content/miniforge3/bin/conda run -n tddpm \
    env PYTHONPATH=/content/tab-ddpm \
    PROJECT_DIR=/content/tab-ddpm \
    python scripts/pipeline.py \
    --config /content/tab-ddpm/exp/pima/ddpm/config.toml \
    --eval

/content/tab-ddpm
----------------------------------------------------------------------------------------------------
loading synthetic data: /content/tab-ddpm/exp/pima/ddpm
Train size: (1000, 8), Val size (108, 8)
{'seed': 42, 'normalization': 'minmax', 'num_nan_policy': None, 'cat_nan_policy': None, 'cat_min_frequency': None, 'cat_encoding': None, 'y_policy': 'default'}
----------------------------------------------------------------------------------------------------
DecisionTreeClassifier
****************************************************************************************************
[val]
{'acc': 0.7407, 'f1': 0.7122, 'roc_auc': 0.7098}
[test]
{'acc': 0.6797, 'f1': 0.6539, 'roc_auc': 0.6568}
Elapsed time: 0:00:00
/content/miniforge3/envs/tddpm/lib/python3.9/site-packages/torch/torch_version.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-